[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_computing/04_vectorization_and_numpy_performance/first_principles.ipynb)

# Topic 04: Vectorization and NumPy Performance

## 1. First-Principles Intuition & Motivation

A contemporary server core executes roughly $10^{11}$ double-precision flops per second and reads roughly $10^{10}$ bytes per second from DRAM. Since one binary64 number occupies 8 bytes, the machine can perform about **80 arithmetic operations in the time it takes to fetch a single number it has not already cached**. That ratio — not instruction counts, not asymptotic complexity — is what decides the runtime of almost every numerical kernel written today.

Layered on top of that hardware fact is a software one: a CPython bytecode loop costs $50$–$200$ nanoseconds per iteration in dispatch, reference counting, and object boxing, while the underlying `add` instruction costs about $0.3$ nanoseconds. A Python loop over an array therefore runs at $\sim 0.5\%$ of the machine's capability, and no amount of micro-optimization inside the loop body changes that.

**Vectorization** is the discipline of arranging computations so that (i) the interpreter is entered once instead of $n$ times, and (ii) the data that crosses the memory bus is reused as many times as possible before being evicted. This notebook derives both halves quantitatively: the layout algebra that determines how data is traversed, and the roofline arithmetic that predicts whether a kernel is limited by flops or by bytes.

### One computation, three implementations

Compute the squared Euclidean distance matrix $D_{ij} = \Vert a_i - b_j \Vert_2^{2}$ for $A \in \mathbb{R}^{m \times d}$, $B \in \mathbb{R}^{n \times d}$.

```python
# (1) Python loops: m*n*d interpreter iterations
D = [[sum((a[k]-b[k])**2 for k in range(d)) for b in B] for a in A]

# (2) Broadcasting: materializes an (m, n, d) temporary
D = ((A[:, None, :] - B[None, :, :])**2).sum(-1)

# (3) BLAS-3 identity: one gemm plus two cheap reductions
D = (A*A).sum(1)[:, None] + (B*B).sum(1)[None, :] - 2.0 * (A @ B.T)
```

All three are the same mathematics. For $m = n = 4000$, $d = 128$ the flop counts are within a factor of two of each other, yet on a typical machine (1) takes minutes, (2) takes seconds and allocates $m n d \times 8 = 16$ GB of temporary memory, and (3) takes tens of milliseconds inside a tuned `gemm`.

The gap has two independent causes:

1. **Interpreter overhead** — (1) pays $\sim 10^{2}$ ns per scalar operation instead of $\sim 10^{-1}$ ns.
2. **Data movement** — (2) writes and re-reads an $m \times n \times d$ array (arithmetic intensity $\approx \frac{1}{8}$ flop/byte, hopelessly memory-bound), while (3) performs $2mnd$ flops on $O(md + nd + mn)$ bytes (intensity $\approx \frac{d}{2}$, comfortably compute-bound).

Version (3) also illustrates the module's recurring trade: it is *less* numerically stable than (2), because $\Vert a \Vert^{2} + \Vert b \Vert^{2} - 2a^{\top}b$ suffers cancellation when $a \approx b$ (Topic 02), and can even return small negative values. Performance and accuracy are decided together, never separately.

## 2. Rigorous Mathematical Definitions & Theorem Statements

### Definition 2.1 (The strided array model)

An `ndarray` is the quadruple $(\text{buffer}, \text{shape}, \text{strides}, \text{itemsize})$, with shape $(n_1, \dots, n_d)$ and strides $(s_1, \dots, s_d)$ in **bytes**. The element at multi-index $(i_1, \dots, i_d)$ lives at byte offset

$$
\mathrm{offset}(i_1, \dots, i_d) = \sum_{k=1}^{d} i_k\, s_k
$$

measured from the buffer's base pointer (plus a fixed base offset for sliced views).

- **C (row-major) order**: $s_d = \text{itemsize}$, $s_k = s_{k+1} n_{k+1}$, so the *last* index varies fastest.
- **Fortran (column-major) order**: $s_1 = \text{itemsize}$, $s_k = s_{k-1}n_{k-1}$, so the *first* index varies fastest.
- **View**: any operation expressible purely as a change of $(\text{shape}, \text{strides}, \text{base offset})$ — slicing with a step, transposing, `reshape` when compatible, `broadcast_to`, `swapaxes`. Cost $O(1)$, no data touched.
- **Copy**: required when the requested layout is not expressible with strides — fancy/boolean indexing, `reshape` across a non-contiguous boundary, `ascontiguousarray`, most dtype casts. Cost $O(\text{size})$ and one full pass over memory.

Consequences worth memorizing: `A.T` is free; `A.T.copy()` is a full transpose; `A[::2]` is free but halves effective bandwidth utilization (every second element of each fetched cache line is discarded); `A[idx]` with an integer array is always a copy and is gather-limited.

### Definition 2.2 (Contiguity, cache lines, and traversal cost)

Memory is transferred in **cache lines** of $L = 64$ bytes ($= 8$ binary64 numbers). A traversal that advances by $s$ bytes per element touches

$$
\text{lines per element} = \begin{cases} \dfrac{\text{itemsize}}{L} & s = \text{itemsize} \; (\text{contiguous}) \\[2mm] 1 & s \ge L \; (\text{strided beyond a line}) \end{cases}
$$

so a stride of $64$ bytes or more transfers $L/\text{itemsize} = 8\times$ more bytes than the algorithm consumes. This is the entire content of the "loop over the fast axis" rule:

$$
\frac{T_{\text{wrong order}}}{T_{\text{right order}}} \; \approx \; \frac{L}{\text{itemsize}} = 8 \quad \text{(binary64, until TLB effects make it worse)}
$$

An array is **C-contiguous** when its strides equal the C-order strides for its shape, i.e. when it can be traversed as a flat buffer. NumPy checks contiguity flags to decide whether a ufunc can use its single-loop fast path or must fall back to a strided (and often non-SIMD) inner loop.

### Theorem 2.3 (Broadcasting rules)

Given operands with shapes $S^{(1)}, \dots, S^{(p)}$, NumPy forms the result shape as follows.

1. **Right-align** all shapes, prepending $1$s to the shorter ones.
2. For each axis $k$, the output extent is $n_k = \max_j n_k^{(j)}$, and the operation is **valid** if and only if every $n_k^{(j)} \in \{1, n_k\}$.
3. An operand axis with extent $1$ is expanded by setting its stride to $\mathbf{0}$ — reading the same element repeatedly, with no data duplication.

**Cost accounting.** Broadcasting is free on the *inputs* (stride-0 trick, $O(1)$ metadata) but the *output* is materialized in full:

$$
\text{bytes written} = \text{itemsize} \times \prod_{k} n_k
$$

Thus `A[:, None, :] - B[None, :, :]` for $A \in \mathbb{R}^{m \times d}$, $B \in \mathbb{R}^{n \times d}$ costs $O(mnd)$ bytes of traffic even though the inputs total $O((m+n)d)$ — a hidden blow-up of factor $\frac{mnd}{(m+n)d} \approx \min(m, n)$.

**Corollary (the reduction-fusion rule).** Whenever a broadcast is immediately reduced, ask whether the reduction can be pushed inside a BLAS call or expressed with `einsum`/`tensordot`, eliminating the intermediate entirely.

### Definition 2.4 (Universal functions)

A **ufunc** is a vectorized C kernel with a fixed type signature that NumPy applies elementwise over broadcast operands. Its properties:

- **Loop selection**: NumPy chooses the inner loop by dtype; if inputs need casting, it allocates buffers and casts in chunks (default buffer size $8192$ elements). Mixed-dtype expressions therefore incur silent extra passes.
- **`out=` and in-place operators**: `np.add(a, b, out=a)` and `a += b` avoid allocating a temporary and avoid one full pass writing then reading it.
- **Reductions and accumulations**: `ufunc.reduce`, `.accumulate`, `.reduceat`, `.outer`. `np.add.reduce` (i.e. `np.sum`) uses **blocked pairwise summation** with block size 128, which is why its error is $O(u\log n)$ rather than $O(un)$ (Topic 02) at no speed cost.
- **SIMD**: the contiguous inner loops are compiled with vector intrinsics (SSE/AVX2/AVX-512, NEON, SVE), processing $4$–$8$ binary64 lanes per instruction. Non-contiguous loops usually fall back to scalar code.

**The temporary-allocation law.** An expression tree with $t$ binary ufunc nodes over arrays of $N$ elements allocates $t - 1$ temporaries and performs

$$
\text{memory traffic} \approx (2t + 1) \, N \times \text{itemsize} \text{ bytes}
$$

(read two, write one, per node) whereas a *fused* implementation reads each input once and writes once: $\approx (p + 1)N \times \text{itemsize}$. Fusing is exactly the job of `numexpr`, `numba`, `torch.compile`, and XLA.

### Definition 2.5 (BLAS levels and arithmetic intensity)

The **arithmetic intensity** of a kernel is

$$
I = \frac{\text{flops performed}}{\text{bytes moved between DRAM and cache}} \quad \left[ \frac{\text{flop}}{\text{byte}} \right]
$$

The three BLAS levels are precisely a classification by intensity (binary64, $8$ bytes per element):

| Level | Example | Flops | Bytes (compulsory) | $I$ |
|---|---|---|---|---|
| 1 — vector–vector | `axpy`: $y \mathrel{+}= \alpha x$ | $2n$ | $24n$ | $\frac{1}{12}$ |
| 1 — vector–vector | `dot`: $x^{\top}y$ | $2n$ | $16n$ | $\frac{1}{8}$ |
| 2 — matrix–vector | `gemv`: $y \mathrel{+}= Ax$ | $2n^{2}$ | $8n^{2} + O(n)$ | $\frac{1}{4}$ |
| 3 — matrix–matrix | `gemm`: $C \mathrel{+}= AB$ | $2n^{3}$ | $\ge 24n^{2}$ | $\frac{n}{12}$ |

Levels 1 and 2 have intensity **independent of $n$** and bounded by a small constant: they are permanently memory-bound, and their runtime is $\text{bytes}/B_{\text{mem}}$ no matter how fast the ALUs are. Level 3 has intensity growing linearly in $n$: it is the only category that can reach peak flops, and only if written to reuse cached blocks (Theorem 2.7).

**Design rule.** Restructure algorithms so that work migrates from level 1/2 to level 3 — batching $k$ matrix–vector products into one matrix–matrix product raises $I$ from $\frac{1}{4}$ to $O(k)$. This is why blocked LU/QR/Cholesky exist, why mini-batching is fast, and why "one big matmul" beats "a loop of small matmuls".

### Theorem 2.6 (Roofline model)

Let $P_{\text{peak}}$ be peak flop rate (flop/s) and $B_{\text{mem}}$ sustainable bandwidth (byte/s). A kernel with arithmetic intensity $I$ attains at most

$$
P(I) = \min\left( P_{\text{peak}}, \; B_{\text{mem}} \cdot I \right)
$$

and its runtime obeys

$$
T \; \ge \; \max\left( \frac{\text{flops}}{P_{\text{peak}}}, \; \frac{\text{bytes}}{B_{\text{mem}}} \right)
$$

The **ridge point** $I^{*} = P_{\text{peak}}/B_{\text{mem}}$ separates the regimes:

$$
I \lt I^{*} \Rightarrow \text{memory-bound}, \qquad I \gt I^{*} \Rightarrow \text{compute-bound}
$$

Representative ridge points: a server CPU with $P_{\text{peak}} = 2$ Tflop/s and $B_{\text{mem}} = 200$ GB/s has $I^{*} = 10$ flop/byte; an A100 GPU with $19.5$ Tflop/s (fp32) and $1.55$ TB/s has $I^{*} \approx 12.6$; with tensor cores at $312$ Tflop/s (fp16) it rises to $I^{*} \approx 200$ — which is why half-precision hardware makes *more* workloads memory-bound, not fewer.

**Diagnostic use.** Compute $I$ on paper, compare with $I^{*}$, and you know before profiling whether to optimize arithmetic (fewer flops, better instruction mix) or data movement (fusion, blocking, dtype, layout). Optimizing the wrong side yields exactly zero.

### Theorem 2.7 (Cache blocking and the communication lower bound)

**Hong–Kung / Irony–Toledo–Tiskin.** Any schedule of the classical $2n^{3}$-flop matrix multiplication on a machine with fast memory of size $M$ words must move at least

$$
Q = \Omega\!\left( \frac{n^{3}}{\sqrt{M}} \right)
$$

words between slow and fast memory. The bound is attained by **blocked (tiled) multiplication**: partition $A, B, C$ into $b \times b$ tiles with

$$
3b^{2} \le M \quad \Longrightarrow \quad b = \left\lfloor \sqrt{M/3} \right\rfloor
$$

so that one tile of each of $A$, $B$, $C$ resides simultaneously in fast memory. Each tile of $C$ is then updated by $n/b$ tile products, giving total traffic

$$
Q_{\text{blocked}} = \Theta\!\left( \frac{n^{3}}{b} \right) = \Theta\!\left( \frac{n^{3}}{\sqrt{M}} \right)
$$

versus $\Theta(n^{3})$ for the naive triple loop — an improvement by the factor $b = \sqrt{M/3}$, which is the achieved reuse per loaded element.

**Arithmetic intensity of the blocked kernel:**

$$
I_{\text{blocked}} = \frac{2n^{3}}{8 \cdot \Theta(n^{3}/b)} = \Theta(b) = \Theta\!\left( \sqrt{M} \right)
$$

so intensity is set by the *cache size*, not by the problem size. Real BLAS implementations (Goto/BLIS) block for three levels simultaneously — L1 for the micro-kernel, L2 for a panel of $A$, L3 for a panel of $B$ — and *pack* panels into contiguous scratch buffers so the micro-kernel sees unit strides.

**Accuracy note (Higham, Ch. 13).** Blocking changes the summation order but not the error model: the computed $\hat{C}$ still satisfies $\hat{C} = AB + \Delta$ with $\vert \Delta \vert \le \gamma_{n}\vert A \vert \vert B \vert$ elementwise. Blocked algorithms are as accurate as unblocked ones — in fact marginally better, since accumulation depth drops from $n$ to $O(b + n/b)$.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Derivation 3.1: Strides, free transposes, and the cost of traversal order

**Setup.** Let $X$ be $m \times n$ binary64 in C order: $\text{strides} = (8n, 8)$.

**Transpose is metadata.** $X^{\top}$ has shape $(n, m)$ and strides $(8, 8n)$ — the same buffer, the tuple reversed. Verify the offset formula is preserved:

$$
\mathrm{offset}_{X^{\top}}(j, i) = 8j + 8n i = \mathrm{offset}_{X}(i, j)
$$

so $X^{\top}[j, i]$ and $X[i, j]$ name the same byte. No data moves; the operation is $O(1)$. Note that $X^{\top}$ is then *Fortran*-contiguous, which is why BLAS calls on transposed operands are free (the `trans` flag simply switches the loop order) while `np.ascontiguousarray(X.T)` is a full $O(mn)$ transpose.

**Traversal cost.** Summing along axis 1 of a C-order array advances by 8 bytes per element: each 64-byte cache line delivers 8 useful values, so bytes moved $= 8mn$. Summing along axis 0 advances by $8n$ bytes: if $8n \ge 64$ (i.e. $n \ge 8$), every element sits on a distinct line and the traffic is

$$
\text{bytes} = 64 \, mn \qquad \text{— an } 8\times \text{ penalty}
$$

For arrays larger than the TLB reach the penalty grows further (page-table walks per access), and measured ratios of $10$–$30\times$ are common.

**The reconciliation.** `X.sum(axis=0)` on a C-ordered array is *not* slow in NumPy, because the reduction is implemented as a loop over rows accumulating into a contiguous output vector — the access pattern is repaired by the loop nesting, not by the layout. The lesson generalizes: **a bad layout is only a problem when the kernel cannot reorder its loops**, which is exactly the case for fancy indexing, elementwise ufuncs on mismatched layouts, and hand-written Python/Numba loops.

```python
# Diagnostics worth internalizing (illustrative):
# X.strides, X.flags['C_CONTIGUOUS'], X.base is None   # view or copy?
# np.shares_memory(a, b)                               # aliasing check
```

### Derivation 3.2: The hidden cost of broadcasting, and how to remove it

**Claim.** Computing $D_{ij} = \Vert a_i - b_j \Vert_2^{2}$ by broadcasting costs $\Theta(mnd)$ memory traffic; the Gram-matrix identity costs $\Theta(mn + (m+n)d)$ traffic and $2mnd$ flops inside `gemm`.

**Broadcast route.** `A[:, None, :] - B[None, :, :]` has shape $(m, n, d)$. Both operands are stride-0-expanded (free), but the difference is materialized: $8mnd$ bytes written, then read again by `**2`, written again, read again by `.sum(-1)`. Total traffic $\approx 4 \times 8mnd = 32mnd$ bytes for $3mnd$ flops:

$$
I_{\text{broadcast}} = \frac{3mnd}{32mnd} \approx 0.094 \; \frac{\text{flop}}{\text{byte}}
$$

— an order of magnitude below any ridge point, so this kernel runs at DRAM speed regardless of hardware. For $m = n = 4000$, $d = 128$ the temporary alone is $16$ GB.

**Algebraic route.** Expand the square:

$$
\Vert a_i - b_j \Vert_2^{2} = \Vert a_i \Vert_2^{2} + \Vert b_j \Vert_2^{2} - 2\, a_i^{\top}b_j
$$

The cross term is the single matrix product $AB^{\top}$ ($2mnd$ flops, level 3), and the two norm vectors cost $O(md + nd)$. Traffic is $8(md + nd + mn)$ bytes:

$$
I_{\text{gemm}} = \frac{2mnd}{8(md + nd + mn)} \approx \frac{d}{4} \quad (m = n)
$$

For $d = 128$ that is $32$ flop/byte — above the CPU ridge point, so the kernel runs at peak.

**The stability caveat.** The identity subtracts $2a^{\top}b$ from $\Vert a \Vert^{2} + \Vert b \Vert^{2}$; when $a \approx b$ these are nearly equal and cancellation (Topic 02, Theorem 2.6) destroys relative accuracy — the computed $D_{ii}$ can be a small negative number. Production code clips at zero and, where exact small distances matter, falls back to the direct form. **This is the module's central trade in miniature: the fast formulation is the ill-conditioned one, and knowing both is the point.**

### Derivation 3.3: Arithmetic intensities of the BLAS levels

Assume binary64 ($8$ bytes) and count *compulsory* traffic — each input read once, each output written once — which is the best case achievable with perfect caching.

**Level 1, `axpy` ($y \mathrel{+}= \alpha x$).** Flops: $2n$ (one multiply, one add per element). Bytes: read $x$ ($8n$), read $y$ ($8n$), write $y$ ($8n$) $= 24n$.

$$
I_{\text{axpy}} = \frac{2n}{24n} = \frac{1}{12} \approx 0.083
$$

**Level 2, `gemv` ($y \mathrel{+}= Ax$).** Flops: $2n^{2}$. Bytes: $A$ dominates at $8n^{2}$; vectors add $O(n)$.

$$
I_{\text{gemv}} = \frac{2n^{2}}{8n^{2}} = \frac{1}{4}
$$

Note this is independent of $n$: **every element of $A$ is used exactly once**, so no cache can help. A `gemv` runs at bandwidth speed forever.

**Level 3, `gemm` ($C \mathrel{+}= AB$, all $n \times n$).** Flops: $2n^{3}$. Compulsory bytes: $8n^{2}$ each for $A$, $B$, read of $C$, and write of $C$ — say $\ge 24n^{2}$.

$$
I_{\text{gemm}} = \frac{2n^{3}}{24n^{2}} = \frac{n}{12}
$$

Now every element participates in $n$ multiply–adds, so the reuse is $\Theta(n)$ *if the schedule keeps it in cache* — which is exactly what Theorem 2.7's blocking guarantees, capping the achievable reuse at $\Theta(\sqrt{M})$ when $n \gg \sqrt{M}$.

**Worked comparison at $n = 1000$, CPU with $I^{*} = 10$:**

| Kernel | Flops | $I$ | Regime | Predicted time at $B = 200$ GB/s, $P = 2$ Tflop/s |
|---|---|---|---|---|
| `axpy` | $2 \times 10^{3}$ | $0.083$ | memory | $24 \times 10^{3}$ B $/\,B_{\text{mem}} = 0.12\;\mu$s |
| `gemv` | $2 \times 10^{6}$ | $0.25$ | memory | $8 \times 10^{6}$ B $/\,B_{\text{mem}} = 40\;\mu$s |
| `gemm` | $2 \times 10^{9}$ | $83$ | compute | $2 \times 10^{9} / P_{\text{peak}} = 1$ ms |

The `gemm` does $1000\times$ the flops of the `gemv` in only $25\times$ the time — the practical meaning of "level 3 is the only place peak performance lives".

### Derivation 3.4: Deriving the optimal block size

**Setup.** Multiply $C = AB$ with all matrices $n \times n$, fast memory (cache) of $M$ words. Partition into $b \times b$ tiles, $t = n/b$ tiles per dimension. The blocked algorithm is

$$
C_{IJ} \mathrel{+}= \sum_{K=1}^{t} A_{IK}B_{KJ}, \qquad I, J = 1, \dots, t
$$

**Step 1 — capacity constraint.** The inner tile product needs $A_{IK}$, $B_{KJ}$, $C_{IJ}$ resident simultaneously:

$$
3b^{2} \le M \quad \Longrightarrow \quad b \le \sqrt{M/3}
$$

**Step 2 — traffic count.** For each of the $t^{2}$ output tiles: read $C_{IJ}$ once ($b^{2}$), write it once ($b^{2}$), and for each of $t$ values of $K$ read $A_{IK}$ and $B_{KJ}$ ($2b^{2}$ each pass). Total words:

$$
Q = t^{2}\left( 2b^{2} + 2tb^{2} \right) = 2\frac{n^{2}}{b^{2}}b^{2} + 2\frac{n^{3}}{b^{3}}b^{2} = 2n^{2} + \frac{2n^{3}}{b}
$$

**Step 3 — optimize.** $Q$ decreases monotonically in $b$, so take $b$ as large as capacity allows, $b = \sqrt{M/3}$:

$$
Q_{\min} = 2n^{2} + \frac{2\sqrt{3}\, n^{3}}{\sqrt{M}} = \Theta\!\left( \frac{n^{3}}{\sqrt{M}} \right)
$$

matching the Hong–Kung lower bound up to a constant — the blocked algorithm is **communication-optimal**.

**Step 4 — numbers.** With a $32$ KiB L1 cache, $M = 4096$ binary64 words, $b = \lfloor\sqrt{4096/3}\rfloor = 36$. Reuse factor $36$ means intensity $I = 2b/8 \approx 9$ flop/byte with respect to the next level down — right at a CPU ridge point, which is not a coincidence: hardware designers size caches so that blocked BLAS-3 lands there. With a $1$ MiB L2, $b \approx 209$; with a $32$ MiB L3, $b \approx 1180$.

**Step 5 — why real BLAS is more complicated.** Goto's algorithm blocks for all levels at once and additionally *packs* $A$-panels and $B$-panels into contiguous, micro-kernel-friendly buffers, so the innermost loop sees unit stride and can be written with SIMD intrinsics and register tiling ($m_r \times n_r$ accumulators held in vector registers). The packing costs one extra pass over the data — negligible against $\Theta(n^{3})$ arithmetic, and it converts every subsequent access into a streaming one.

### Derivation 3.5: Roofline placement of real kernels

Given $P_{\text{peak}}$, $B_{\text{mem}}$, $I^{*} = P_{\text{peak}}/B_{\text{mem}}$, the recipe is: count flops, count *compulsory* bytes, divide, compare.

**Example A — elementwise activation $y = \mathrm{GELU}(x)$**, $N$ binary32 elements. Flops $\approx 10N$ (a `tanh` approximation), bytes $= 4N$ read $+\,4N$ write $= 8N$:

$$
I = \frac{10N}{8N} = 1.25 \; \ll \; I^{*}
$$

Memory-bound. Consequence: **fusing** the activation into the preceding matmul's epilogue removes an entire round trip and is nearly free; adding more arithmetic inside the fused kernel costs nothing.

**Example B — LayerNorm over a $(B, T, H)$ tensor.** Naively three passes (mean, variance, normalize) $\Rightarrow 6N$ bytes read/written per pass. A single-pass Welford implementation (Topic 02) reads once and writes once: $2\times$ less traffic, $2\times$ faster, *and* numerically better conditioned. Rare and delightful: the fast version is also the accurate one.

**Example C — a transformer block, $B$ tokens, hidden $H$, head dim $d$.** Projections and MLP are `gemm`s with $I = \Theta(\min(B, H))$ — compute-bound for large batches, memory-bound for $B = 1$ decoding (which is why autoregressive inference is bandwidth-limited and why KV-cache size dominates latency). Attention's softmax and masking are elementwise over a $B \times B$ score matrix with $I \approx 1$: memory-bound, $\Theta(B^{2})$ traffic. **FlashAttention** is exactly Theorem 2.7 applied here — tile the score matrix, keep tiles in SRAM, fuse softmax into the tile loop with an online (streaming) normalizer, never materialize the $B \times B$ array. The speedup comes entirely from traffic reduction, not from fewer flops.

**Example D — sparse matrix–vector product** with $\mathrm{nnz}$ nonzeros in CSR: $2\,\mathrm{nnz}$ flops, $\ge 12\,\mathrm{nnz}$ bytes (value + column index) plus irregular gathers into $x$:

$$
I \le \frac{1}{6}
$$

Permanently memory-bound and worse in practice due to random access. This is why sparse methods win only when $\mathrm{nnz} \ll n^{2}$ by orders of magnitude, and why blocked/structured sparsity (2:4, block-sparse) is preferred in ML: it restores contiguity.

### Derivation 3.6: `einsum` and the contraction-order problem

`einsum` specifies a tensor contraction by index notation, e.g. `"ij,jk,kl->il"` for $A B C$. Its *cost* depends entirely on the order in which pairs are contracted.

**Two orders, two complexities.** For $A \in \mathbb{R}^{n \times n}$, $B \in \mathbb{R}^{n \times n}$, $C \in \mathbb{R}^{n \times m}$:

$$
(AB)C: \; 2n^{3} + 2n^{2}m \text{ flops}, \qquad A(BC): \; 2n^{2}m + 2n^{2}m = 4n^{2}m \text{ flops}
$$

For $m \ll n$ the right-to-left order is cheaper by $\Theta(n/m)$. Worse, a *naive* single nested loop over all indices — which `einsum` used before `optimize` existed — costs $\Theta(n^{3}m)$, asymptotically worse than either pairwise order.

**The rule.** `np.einsum(..., optimize=True)` runs a greedy/optimal search (exhaustive for up to 4 operands, greedy beyond) over contraction trees, minimizing estimated flops, and dispatches each pairwise contraction to `tensordot` $\to$ `gemm` when the index pattern permits. Always pass it for 3 or more operands; for two operands it is usually a no-op.

**When `einsum` beats explicit reshapes.** It expresses contractions that would otherwise need `transpose` + `reshape` + `matmul` + `reshape`, each potentially copying. When the pattern maps to `gemm` after at most a stride permutation, `einsum` reaches BLAS speed with no temporaries.

**When it loses.** Patterns with repeated output indices (e.g. `"ii->i"`, batched diagonals), or where the required permutation forces a physical copy, run in `einsum`'s own strided kernel — typically $5$–$50\times$ slower than `gemm`. Diagnose with `np.einsum_path(...)`, which prints the chosen tree, the intermediate shapes, and the flop estimate.

```python
# Attention scores without materializing a permutation:
#   scores = np.einsum('bqhd,bkhd->bhqk', Q, K, optimize=True)
# Inspect the plan before trusting it:
#   print(np.einsum_path('bqhd,bkhd->bhqk', Q, K, optimize='optimal')[1])
```

### Derivation 3.7: What a benchmark actually measures

A timing measurement is a random variable

$$
T_{\text{measured}} = T_{\text{kernel}} + T_{\text{overhead}} + \varepsilon
$$

where $T_{\text{overhead}}$ collects timer resolution, first-touch page faults, allocator behaviour and JIT compilation, and $\varepsilon$ collects interrupts, frequency scaling, and co-tenant interference. Three facts follow.

**1. Use `min`, not `mean`, for kernel throughput.** All contaminating terms are non-negative: $\varepsilon \ge 0$ almost always (nothing makes a computation *faster* than its uncontended execution). The minimum over repetitions is therefore the best estimator of $T_{\text{kernel}}$, while the mean estimates $T_{\text{kernel}} + \mathbb{E}[\varepsilon]$ — a different, machine-load-dependent quantity. Report the mean only when modelling *service latency*, where interference is part of the product.

**2. State the cache state.** Repeating a kernel on the same small array measures *hot-cache* performance, which can exceed cold-DRAM performance by $10\times$. Choose the regime deliberately: arrays $\gg$ L3 size for streaming benchmarks; arrays $\ll$ L1 for instruction-throughput benchmarks; and rotate through several buffers to avoid accidental reuse.

**3. Amortize the timer.** With a clock resolution of $\sim 100$ ns, a $200$ ns kernel must be repeated in an inner loop — but a compiler or NumPy may then hoist or cache the work. `timeit` handles the loop; the defence against hoisting is to consume the result.

**Checklist.**

- Warm up (allocate, touch pages, trigger JIT) before timing.
- Repeat $\ge 7$ times, each with enough inner iterations to exceed $10\,\text{ms}$.
- Report `min` and the spread; a spread above $10\%$ means the environment is unreliable.
- Fix threads: `OMP_NUM_THREADS` / `threadpoolctl` — an unpinned BLAS makes results irreproducible.
- Convert to a *rate* (Gflop/s or GB/s) and compare against the roofline; an absolute time in seconds tells you nothing about whether optimization is still possible.
- For GPUs, synchronize before stopping the clock; kernel launches are asynchronous.

## 4. Computational & Algorithmic Insights

### 4.1 A decision procedure for slow array code

1. **Measure the intensity, not the profile.** Count flops and compulsory bytes; place the kernel on the roofline. Everything below depends on which side it lands.
2. **If memory-bound:**
   - Eliminate temporaries: `out=`, in-place operators, `np.add(a, b, out=c)`.
   - Fuse passes: `numexpr` (multi-threaded expression fusion, no compilation), `numba.njit(fastmath=True)`, `torch.compile`, JAX/XLA.
   - Shrink the data: float32 instead of float64 halves the traffic and therefore roughly halves the time (check the accuracy budget with Topic 03's digit rule first).
   - Improve locality: match traversal to layout, tile large reductions, use `np.ascontiguousarray` *once* if a kernel will traverse the array many times.
3. **If compute-bound:**
   - Make sure the work is in BLAS-3, threaded, and using the right dtype.
   - Check that the linked BLAS is a tuned one (OpenBLAS/MKL/BLIS), not the reference netlib build; `np.show_config()` reveals it.
   - Increase batch dimensions to convert level-2 into level-3 work.
4. **If neither helps, the bottleneck is elsewhere**: Python-level dispatch (use fewer, larger calls), allocation churn (preallocate and reuse buffers), or I/O (memory-map, prefetch, decode in parallel).

### 4.2 Vectorization patterns worth knowing by name

| Pattern | Slow form | Fast form | Why |
|---|---|---|---|
| Accumulate over rows | Python `for` loop | `arr.sum(axis=1)` | one C loop, pairwise summation, SIMD |
| Conditional assignment | loop with `if` | `np.where(mask, a, b)` | branchless, vectorized |
| Pairwise distances | broadcast difference | Gram identity + `gemm` | level 2 $\to$ level 3 |
| One-hot then matmul | dense $XW$ with one-hot $X$ | `W[idx]` (fancy index) | skips $\Theta(nV)$ of multiply-by-zero |
| Repeated small matmuls | loop of `gemm` | batched `matmul` / `einsum` | amortizes launch and packing overhead |
| Rolling statistics | loop | `cumsum` differences or `sliding_window_view` | strided views, no copy |
| Gather–scatter accumulation | loop with `+=` | `np.add.at` or `np.bincount` | correct with duplicate indices, single pass |
| Polynomial evaluation | explicit powers | Horner via `np.polynomial` | fewer flops, better conditioned |
| Boolean filtering in a loop | append to list | mask once, index once | one allocation, contiguous result |
| Repeated concatenation | `np.concatenate` in a loop | preallocate + assign slices | avoids $O(n^{2})$ copying |

Two anti-patterns deserve explicit mention: `np.vectorize` is **not** vectorization (it is a Python loop with a convenient signature), and `np.apply_along_axis` likewise. Both are readability tools with loop performance.

### 4.3 When NumPy is the wrong tool

- **Element-dependent control flow** (early exits, irregular recursion): NumPy must compute all branches; use Numba or Cython.
- **Long chains of cheap elementwise ops**: each is a separate memory pass; use `numexpr`, Numba, or a compiled framework that fuses.
- **Small arrays in a hot loop** ($n \lesssim 100$): per-call Python overhead ($\sim 1\;\mu$s) dominates; batch them, or drop into a compiled kernel.
- **Sparse or irregular structure**: use `scipy.sparse` (and expect memory-bound performance, Derivation 3.5D).
- **Data larger than RAM**: use memory-mapped arrays, Dask, or chunked processing; the roofline analysis then applies with disk bandwidth in place of $B_{\text{mem}}$, and $I^{*}$ rises by three orders of magnitude.
- **GPU-resident workloads**: CuPy, PyTorch, JAX. The model carries over unchanged — only $P_{\text{peak}}$, $B_{\text{mem}}$, and the cache sizes change, plus the addition of a PCIe/NVLink transfer term that is usually the true bottleneck for small kernels.

## 5. Real-World Physics & AI/ML Applications

### 5.1 Why deep learning is a batching argument

A single forward pass of a linear layer on one example is $y = Wx$: a `gemv`, intensity $\frac{1}{4}$, running at DRAM speed and using a fraction of a percent of a GPU's flops. Stack $B$ examples into a matrix and it becomes $Y = WX$: a `gemm` with intensity $\Theta(\min(B, H))$, compute-bound for $B$ of a few hundred. **The entire economics of GPU training rests on this conversion of level-2 work into level-3 work** — it is why batch size is a systems parameter as much as a statistical one, and why gradient accumulation (many small batches) is slower than one large batch even at identical flop counts.

The same argument runs in reverse at inference time: autoregressive decoding generates one token per step, so each step is a `gemv` against the weights plus a KV-cache read. Decoding is therefore **memory-bandwidth-bound**, its speed set by $\frac{\text{model bytes}}{B_{\text{mem}}}$, which is why quantization to int8/int4 gives near-linear latency wins (fewer bytes) while adding arithmetic (dequantize on the fly) is free, and why speculative decoding and continuous batching exist — both are devices for restoring a batch dimension.

### 5.2 FlashAttention as textbook cache blocking

Standard attention computes $S = QK^{\top}/\sqrt{d}$ ($B \times B$), applies softmax row-wise, then $O = PV$. The score matrix is materialized in HBM: $\Theta(B^{2})$ writes plus $\Theta(B^{2})$ reads for the softmax plus more for the second matmul. Since the softmax itself is elementwise ($I \approx 1$), the whole layer is memory-bound and scales as $\Theta(B^{2})$ in *traffic*, which for long contexts dominates the $\Theta(B^{2}d)$ arithmetic.

FlashAttention tiles $Q$, $K$, $V$ into blocks sized to on-chip SRAM (precisely Theorem 2.7's capacity constraint $3b^{2} \le M$), computes each score tile in registers, and applies an **online softmax** — the streaming max-and-normalizer update from Topic 02's log-sum-exp trick — so the $B \times B$ matrix never exists in memory. The flop count is unchanged (indeed slightly higher, due to recomputation in the backward pass), yet wall-clock improves by $2$–$4\times$: pure traffic reduction. It is simultaneously a *numerical* result, since the online normalizer keeps the exponentials in range (Topic 05).

### 5.3 Physics: stencils, N-body, and the memory wall

Finite-difference PDE solvers apply a stencil $u^{n+1}_{i} = \sum_{k} c_k u^{n}_{i+k}$: a handful of flops per grid point against $8$ bytes loaded, so $I = O(1)$ and the solver runs at bandwidth speed. Decades of HPC engineering address exactly this: **spatial blocking** (tile the grid so a tile's halo stays in cache), **temporal blocking / time skewing** (advance several time steps within one cache-resident tile, raising $I$ proportionally), and cache-oblivious recursive traversals that achieve near-optimal traffic without tuning tile sizes.

Direct N-body is the opposite extreme: $\Theta(N^{2})$ force evaluations on $\Theta(N)$ data gives $I = \Theta(N)$, deeply compute-bound — which is why GPUs reach near-peak on brute-force N-body while the *faster* algorithms (Barnes–Hut $\Theta(N\log N)$, FMM $\Theta(N)$) are irregular, pointer-chasing, and memory-bound, and only win at large $N$. The general lesson: **reducing flops often converts a compute-bound kernel into a memory-bound one, and the asymptotically better algorithm can be slower until the crossover.**

### 5.4 Data pipelines, dtypes, and the cost of a copy

In production ML training, the model is frequently *not* the bottleneck: JPEG decoding, augmentation, tokenization, and host-to-device transfer are. Each is analyzable with the same tools:

- **Decoding and augmentation** are elementwise, $I \approx 1$, memory-bound — parallelize across cores and fuse operations rather than optimizing arithmetic.
- **Host-to-device copies** run at PCIe bandwidth ($\sim 25$ GB/s for Gen4 x16) versus HBM's $\sim 2$ TB/s: an $80\times$ gap. Pinned (page-locked) memory and asynchronous copies overlapped with compute are the standard fixes.
- **dtype selection is a bandwidth decision.** Storing activations in bf16 halves traffic against fp32; storing image batches as uint8 and casting on the GPU cuts the transfer by $4\times$ against float32. Topic 05 supplies the accuracy analysis that says when this is safe.
- **A silent copy is a silent memory pass.** `np.ascontiguousarray`, an unexpected dtype promotion (`int32 + float64` $\to$ `float64`), a non-contiguous slice fed to a BLAS call, `torch.Tensor.contiguous()` after a `permute` — each costs a full sweep of the array. Profiling allocation counts often finds more time than profiling arithmetic.

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source | Location |
|---|---|---|
| Strided array model, views, broadcasting | Harris et al., *Array programming with NumPy*, Nature 585 (2020) | Fig. 1–2, Box 1 |
| Ufunc machinery, buffering, type resolution | van der Walt, Colbert & Varoquaux, CiSE 13(2) (2011) | Sec. 3 |
| Roofline model, ridge point | Williams, Waterman & Patterson, CACM 52(4) (2009) | Sec. 2–3 |
| BLAS levels and data reuse | Golub & Van Loan, *Matrix Computations* (2013) | Sec. 1.1–1.5 |
| Blocked `gemm`, packing, micro-kernel | Goto & van de Geijn, ACM TOMS 34(3) (2008) | Sec. 3–4 |
| Communication lower bound $\Omega(n^{3}/\sqrt{M})$ | Hong & Kung (1981); Irony, Toledo & Tiskin (2004) | — |
| Cache-oblivious recursive blocking | Frigo, Leiserson, Prokop & Ramachandran, FOCS (1999) | Sec. 2, 6 |
| Cache hierarchy, prefetch, TLB | Drepper, *What Every Programmer Should Know About Memory* (2007) | Sec. 3, 6 |
| Memory hierarchy quantitative model | Hennessy & Patterson, *Computer Architecture* (2019) | Ch. 2, App. B |
| Error bounds of blocked algorithms | Higham, *Accuracy and Stability of Numerical Algorithms* (2002) | Ch. 13 |
| Contraction-order optimization | Daniel, Gray et al., `opt_einsum` (JOSS 2018) | — |
| JIT compilation for unvectorizable loops | Lam, Pitrou & Seibert, *Numba*, LLVM-HPC (2015) | — |
| Tiled attention with online softmax | Dao et al., *FlashAttention* (NeurIPS 2022) | Sec. 3 |
| dtype as a bandwidth decision | Micikevicius et al., *Mixed Precision Training* (ICLR 2018) | Sec. 3 |

**Continue to** [Topic 05: Numerical Stability in Deep Learning](../05_numerical_stability_in_deep_learning/README.md) — where the accuracy analysis of Topics 01–03 and the performance analysis of this module meet, and low-precision arithmetic forces both to be solved at once.